# **Clinical Trial Enrichment: Run 3 (Protocol Strategist)**
---
This notebook orchestrates the third stage of the clinical enrichment pipeline. While previous runs focused on *what* disease is being treated (Run 1) and *what* drug is being used (Run 2), Run 3 identifies the **Rules of the Game**—the strategic and operational design of the trial. 

It integrates results from Runs 1 and 2 with raw AACT structural data to enable the LLM to categorize trials according to the **'Clean 7'** schema: Endpoint Rigor, Endpoint Structure, Comparator Benchmark, Strategic Ambition, Administration Complexity, Innovation Tier, and Adaptive Design.

# **0. Master Control: Reset & Configuration**
Define the execution mode, sampling parameters, and reset artifacts upfront. This ensures reproducibility and prevents the accidental merging of stale data.

In [1]:
# [CONTROL] Set to True to enable deletion of existing input contexts
RESET_PRODUCED_FILES = True

# [CONFIG] Toggle between full run and test sample
IS_PRODUCTION = True  # Set to True for full run
SAMPLE_SIZE = 500
RANDOM_STATE = 40

import os
files_to_delete = [
        '../data/llm_in_03.csv',
    ]

if RESET_PRODUCED_FILES:
    for f in files_to_delete:
        if os.path.exists(f):
            os.remove(f)
            print(f"> Deleted: {f}")
    print("> Reset complete.")
else:
    print("> Reset skipped.")

> Reset complete.


# **1. Environment Setup & Configuration**
Initialize libraries and project-specific utilities. We import the `day_zero_reconstructor` to sanitize linguistic context (removing future results/timestamps) and define path constants.

In [2]:
import pandas as pd
import os
import csv
import re
import sys
import json
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()
sys.path.append('..')
from src.prep.text_cleaning import day_zero_reconstructor

DATA_PATH = '../data/'
OUTPUT_PATH = '../data/processed'
NL = chr(10)

# [STEP 5] Utility function for robust CSV loading with specific clinical formatting
def safe_load(filename, cols=None):
    full_path = os.path.join(DATA_PATH, filename)
    # Match Strategy A: PERFECT from data_loader_clinpred.py
    params_perfect = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": csv.QUOTE_MINIMAL, "low_memory": False, "on_bad_lines": "warn"
    }
    # Match Strategy B: ROBUST
    params_robust = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": 3, "low_memory": False, "on_bad_lines": "warn"
    }
    try:
        return pd.read_csv(full_path, usecols=cols, **params_perfect)
    except:
        return pd.read_csv(full_path, usecols=cols, **params_robust)

print("> SUCCESS: Environment Ready.")


> SUCCESS: Environment Ready.


# **2. Anchor Alignment: Loading Runs 1 & 2**
Run 3 'inherits' intelligence from the previous stages. We load the Indication/TA (Run 1) and Molecular Targets/Precedent (Run 2) to provide the LLM with a complete picture of the trial's clinical and biological context.

In [3]:
# [STEP 1] Load Run 1 Results (Indication & TA)
df_run1 = pd.read_csv(os.path.join(OUTPUT_PATH, 'llm_out_01.csv'), usecols=['nct_id', 'gbd_indication_name', 'therapeutic_area', 'line_of_therapy', 'patient_severity'])

if IS_PRODUCTION:
    target_df = df_run1.copy()
    print(f"> PRODUCTION MODE: Processing all {len(target_df)} trials.")
else:
    target_df = df_run1.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
    print(f"> TESTING MODE: Processing {SAMPLE_SIZE} random trials.")

run1_lookup = target_df.set_index('nct_id').to_dict('index')
target_ids = target_df['nct_id'].tolist()

# [STEP 2] Load Run 2 Results (Molecular & Biomarkers)
df_run2 = pd.read_csv(os.path.join(OUTPUT_PATH, 'llm_out_02.csv'), usecols=['nct_id', 'alpha_drug_name', 'molecular_targets', 'target_precedent', 'biomarker_stratification'])
run2_lookup = df_run2[df_run2['nct_id'].isin(target_ids)].set_index('nct_id').to_dict('index')

print(f"> Loaded {len(target_ids)} trials with Run 1 and Run 2 data.")

> PRODUCTION MODE: Processing all 29557 trials.
> Loaded 29557 trials with Run 1 and Run 2 data.


# **3. AACT Structural Evidence Harvesting**
To judge operational risk, we harvest primary outcomes (Rigor), design info (Ambition), intervention details (Complexity), and brief summaries (Adaptive/Intent). All data is indexed into fast-access dictionaries.

In [4]:
print(">>> Harvesting Enhanced Structural Evidence...")

# [STEP 1] Core Study Metadata (Title, Phase, Date, Arms)
df_studies = safe_load('studies.txt', cols=['nct_id', 'official_title', 'brief_title', 'start_date', 'phase', 'number_of_arms'])
df_studies['official_title'] = df_studies['official_title'].fillna(df_studies['brief_title'])
studies_lookup = df_studies[df_studies['nct_id'].isin(target_ids)].set_index('nct_id').to_dict('index')

# [STEP 2] Primary Outcome Measures + Timeframes (Critical for Rigor)
df_outcomes = safe_load('design_outcomes.txt', cols=['nct_id', 'outcome_type', 'measure', 'time_frame'])
primary_outcomes_lookup = defaultdict(list)
for _, row in df_outcomes[df_outcomes['nct_id'].isin(target_ids)].iterrows():
    if str(row['outcome_type']).strip().upper() == 'PRIMARY':
        m = day_zero_reconstructor(str(row['measure']), "outcome")
        t = day_zero_reconstructor(str(row['time_frame']), "timeframe")
        primary_outcomes_lookup[row['nct_id']].append(f"TITLE: {m} | TIMEFRAME: {t}")

# [STEP 3] Design Information (Allocation, Masking, etc.)
df_designs = safe_load('designs.txt', cols=['nct_id', 'allocation', 'intervention_model', 'primary_purpose', 'masking'])
designs_lookup = df_designs[df_designs['nct_id'].isin(target_ids)].set_index('nct_id').to_dict('index')

# [STEP 4] Intervention Names + Descriptions (Critical for Complexity)
df_int = safe_load('interventions.txt', cols=['nct_id', 'id', 'intervention_type', 'name', 'description'])
df_int_others = safe_load('intervention_other_names.txt', cols=['nct_id', 'intervention_id', 'name'])
synonym_map = defaultdict(list)
for _, row in df_int_others.iterrows():
    synonym_map[row['intervention_id']].append(row['name'])

int_lookup = defaultdict(list)
for _, row in df_int[df_int['nct_id'].isin(target_ids)].iterrows():
    others = synonym_map.get(row['id'], [])
    other_str = f" (Aliases: {', '.join(others)})" if others else ""
    desc = day_zero_reconstructor(row['description'], "intervention") if pd.notna(row['description']) else "No description"
    int_lookup[row['nct_id']].append(f"NAME: {row['name']}{other_str} [{row['intervention_type']}]" + NL + f"DESC: {desc}")

# [STEP 5] Brief Summaries (Critical for Adaptive/Intent)
df_sum = safe_load('brief_summaries.txt', cols=['nct_id', 'description'])
sum_lookup = {row['nct_id']: row['description'] for _, row in df_sum[df_sum['nct_id'].isin(target_ids)].iterrows()}

# [STEP 6] Lead Sponsors (Standardization focus)
df_sponsors = safe_load('sponsors.txt', cols=['nct_id', 'lead_or_collaborator', 'name'])
leads = df_sponsors[df_sponsors['lead_or_collaborator'].str.lower() == 'lead'][['nct_id', 'name']]
sponsor_lookup = {row['nct_id']: row['name'] for _, row in leads[leads['nct_id'].isin(target_ids)].iterrows()}

print("> Enhanced Structural Evidence Harvesting Complete.")

>>> Harvesting Enhanced Structural Evidence...
> Enhanced Structural Evidence Harvesting Complete.


# **4. Context Assembly & Linguistic Diet Enforcement**
The final step iterates through the cohort and assembles the specialized Run 3 context. We enforce a strict 'Linguistic Diet' for each field to ensure high-concurrency performance while prioritizing the most relevant design clues.

In [5]:
results = []
for nct_id in target_ids:
    r1 = run1_lookup.get(nct_id, {})
    r2 = run2_lookup.get(nct_id, {})
    s = studies_lookup.get(nct_id, {})
    d = designs_lookup.get(nct_id, {})

    # Inherited Intelligence (Run 1 & 2)
    ta = r1.get('therapeutic_area', 'Unclassified')
    lot = r1.get('line_of_therapy', 'Unknown')
    sev = r1.get('patient_severity', 'Unknown')
    alpha_drug = r2.get('alpha_drug_name', 'Unknown')
    sponsor = sponsor_lookup.get(nct_id, 'Unknown')

    # Structural Design Clues
    start_date = s.get('start_date', '2000-01-01')
    start_year = str(pd.to_datetime(start_date).year)
    title = day_zero_reconstructor(s.get('official_title', 'Unknown'), "title")[:1000]
    design_str = f"Arms: {s.get('number_of_arms', 'N/A')}, Allocation: {d.get('allocation', 'N/A')}, Model: {d.get('intervention_model', 'N/A')}, Purpose: {d.get('primary_purpose', 'N/A')}, Masking: {d.get('masking', 'N/A')}"[:250]

    # Enhanced Outcomes (Rigor focus)
    outcomes = primary_outcomes_lookup.get(nct_id, ["No primary outcome listed"])
    outcome_str = (NL + "---" + NL).join(outcomes)[:1000]

    # Enhanced Interventions (Complexity focus)
    ints_list = int_lookup.get(nct_id, ["No intervention details"])
    ints_string = (NL + "---" + NL).join(ints_list)[:1000]

    # Summary (Adaptive/Ambition focus)
    summary = day_zero_reconstructor(sum_lookup.get(nct_id, ""), "summary")[:2500]

    # [STRATEGY: SURGICAL CONTEXT REDUCTION]
    # Removed: MOLECULAR_TARGETS, TARGET_PRECEDENT, BIOMARKER_STRATIFIED
    context_body = f"""[TRIAL_START]
[NCT_ID]: {nct_id}
[START_YEAR]: {start_year}
[PHASE]: {s.get('phase', 'N/A')}
[ASSIGNED_INDICATION]: {r1.get('gbd_indication_name', 'Unknown')}
[ASSIGNED_THERAPEUTIC_AREA]: {ta}
[OFFICIAL_TITLE]: {title}
[AGENT]: {alpha_drug}
[SPONSOR]: {sponsor}
[ASSIGNED_LINE_OF_THERAPY]: {lot}
[ASSIGNED_PATIENT_SEVERITY]: {sev}
[DESIGN]: {design_str}

[PRIMARY_ENDPOINTS_DETAIL]:
{outcome_str}

[INTERVENTION_DETAILS_ENHANCED]:
{ints_string}

[PROTOCOL_SUMMARY]:
{summary}
[TRIAL_END]"""

    results.append({"nct_id": nct_id, "context": context_body})

# Export to llm_in_03.csv
pd.DataFrame(results).to_csv(os.path.join(DATA_PATH, 'llm_in_03.csv'), index=False)
print(f"> Success: Assembled LEAN Run 3 context for {len(results)} trials.")


> Success: Assembled LEAN Run 3 context for 29557 trials.


---
## **Next Step: Run Enrichment Orchestrator**
After generating the context CSV in this notebook, shift to your terminal and execute the following command to start the LLM processing stage:

```bash
python3 src/prep/llm_in_03_run.py
```